# TP6 – Testing and Continuous Integration with Pytest (Titanic Project)

## 🎯 Objectives

By the end of this TP, you should be able to:
- Understand the role of testing in MLOps and CI
- Write unit tests using pytest
- Use fixtures and conftest.py to avoid code duplication
- Apply mocking to isolate parts of your code
- Run tests locally in a way compatible with Continuous Integration

This TP is applied to the Titanic project developed in previous sessions.

## 1️⃣ Why testing and CI matter in MLOps

When developers talk about Continuous Integration (CI), they usually think of automatic testing.

CI ensures that:
- every change to the codebase is checked,
- bugs are detected early,
- collaboration is safer.

However:

⚠ CI does not remove bugs by itself. CI only automates the execution of tests.

> "Continuous Integration doesn't get rid of bugs, but it does make them dramatically easier to find and remove." — Martin Fowler

In MLOps, testing is even more important because:
- ML systems depend on data,
- behavior can change without code changes,
- errors can be silent.

In this TP, we focus on unit tests, the first layer of CI.

## 2️⃣ Installing pytest

### Using pip

```bash
pip install pytest
```

### Using uv

```bash
uv add pytest
```

👉 pytest is a development dependency. It should not be required to run the model, only to test it.

## 3️⃣ Test discovery rules (important)

Pytest automatically finds and runs tests if:
- test files are named: `test_*.py`
- test functions are named: `test_*`

Example:

```python
def test_example():
    assert 1 + 1 == 2

```

## 4️⃣ Test structure for the Titanic project

Recommended structure:

```
project/
├── src/
│   ├── data.py
│   ├── features.py
│   ├── model.py
│   └── train.py
├── data/
│   ├── train.csv
│   └── test.csv
├── tests/
│   ├── __init__.py
│   ├── conftest.py
│   ├── test_data.py
│   ├── test_features.py
│   ├── test_model.py
│   └── test_training.py
```

## 5️⃣ Shared configuration: tests/__init__.py

Example `tests/__init__.py`:

```python
import os

_TEST_ROOT = os.path.dirname(__file__)
_PROJECT_ROOT = os.path.dirname(_TEST_ROOT)
_PATH_DATA = os.path.join(_PROJECT_ROOT, "data/raw")

# This allows tests to reliably access project data.
```

## 6️⃣ Basic data tests – test_data.py

```python
import os
import pandas as pd
import pytest
from tests import _PATH_DATA

DATA_PATH = os.path.join(_PATH_DATA, "titanic.csv")

@pytest.mark.skipif(
    not os.path.exists(DATA_PATH),
    reason="Training data not found"
)
def test_train_data_loading():
    df = pd.read_csv(DATA_PATH)
    assert len(df) > 0, "Training dataset is empty"
    assert "Survived" in df.columns, "Target column missing"
    assert "Age" in df.columns
    assert "Fare" in df.columns
```

✅ This test protects against:
- missing files,
- broken pipelines,
- schema drift.

## 7️⃣ Introducing fixtures (avoiding repetition)

### Problem

Many tests require the same toy Titanic dataset.

### Solution: pytest fixtures

Create `tests/conftest.py`:

```python
import pytest
import pandas as pd
import numpy as np

@pytest.fixture
def titanic_sample():
    X = pd.DataFrame({
        "Age": [22, 38, 26, 35],
        "Fare": [7.25, 71.28, 7.92, 53.1],
        "Sex": ["male", "female", "female", "female"],
        "Embarked": ["S", "C", "S", "S"],
    })
    y = np.array([0, 1, 1, 1])
    return X, y

```

✔ Fixtures:
- are automatically injected,
- reduce copy-paste,
- make tests clearer.

## 8️⃣ Model tests – test_model.py

```python
from src.model import build_pipeline

def test_pipeline_fit_and_predict(titanic_sample):
    X, y = titanic_sample
    pipe = build_pipeline(n_trees=5)
    pipe.fit(X, y)
    preds = pipe.predict(X)
    assert preds.shape == (len(X),), "Prediction shape is incorrect"

```

## 9️⃣ Testing errors and defensive programming

Good ML code should fail fast and explicitly.

Example in code:

```python
def build_pipeline(n_trees):
    if n_trees <= 0:
        raise ValueError("n_trees must be positive")
    # ... rest of the function

```

Corresponding test:

```python
import pytest
from src.model import build_pipeline

def test_invalid_n_trees():
    with pytest.raises(ValueError, match="n_trees must be positive"):
        build_pipeline(0)

```

## 🔟 Mocking: isolating your tests

### Why mocking?

Unit tests should not:
- train heavy models,
- call external APIs,
- read large files.

Mocking replaces real dependencies with fake ones.

### Example: mocking file loading

```python
from unittest.mock import patch
import pandas as pd
from src.data import load_data

def test_load_data_mocked():
    fake_df = pd.DataFrame({"A": [1, 2, 3]})
    with patch("pandas.read_csv", return_value=fake_df):
        df = load_data("fake/path.csv")
        assert df.equals(fake_df)

```

✔ This test:
- runs instantly,
- isolates logic,
- is CI-friendly.

## Mocking model training

```python
from unittest.mock import MagicMock
from sklearn.ensemble import RandomForestClassifier

def test_model_fit_called(titanic_sample):
    X, y = titanic_sample
    model = RandomForestClassifier()
    model.fit = MagicMock()
    model.fit(X, y)
    model.fit.assert_called_once()

```

## Running all tests

```bash
pytest tests/
```

If all tests pass locally, they are ready for CI.

# Measuring Test Coverage

## 1️⃣ What is test coverage?

Test coverage measures which parts of your code are executed when your tests run.

It answers the question:

> "When I run my test suite, how much of my code actually runs?"

Coverage is usually expressed as a percentage of lines executed.

⚠️ **Important clarification**

- High coverage does NOT mean good tests
- Low coverage does NOT automatically mean bad code
- Coverage is an indicator, not a goal.

## 2️⃣ Why coverage matters in MLOps

In MLOps projects:
- code evolves quickly,
- pipelines are refactored,
- data processing logic changes often.

Coverage helps you:
- detect untested areas,
- identify dead or unreachable code,
- avoid false confidence in CI.

Coverage answers:

> "If this file changes, will CI notice?"

## 3️⃣ Installing coverage

### Using pip

```bash
pip install coverage
```

### Using uv

```bash
uv add coverage
```

📌 coverage is a development dependency.

## 4️⃣ Running tests with coverage

Instead of:

```bash
pytest tests/
```

Run:

```bash
coverage run -m pytest tests/
```

Then generate a report:

```bash
coverage report -m
```

## 5️⃣ Interpreting the coverage report

Example output:

```
Name          Stmts   Miss  Cover   Missing
------------------------------------------------------------
src/data.py      20      2    90%   45-46
src/features.py  34     12    65%   70-85
src/model.py     28      5    82%   91-96
src/train.py     15      0   100%
------------------------------------------------------------
TOTAL            97     19    80%
```

### Columns explained

| Column | Meaning |
|--------|---------|
| Stmts | Total executable lines |
| Miss | Lines never executed |
| Cover | Coverage percentage |
| Missing | Exact missing line numbers |

## 6️⃣ How to use coverage correctly

### ✅ Good use of coverage

- Identify critical logic not tested
- Improve tests for:
  - feature engineering
  - preprocessing logic
  - error handling
- Detect dead code that can be removed

### ❌ Bad use of coverage

- Chasing 100% coverage blindly
- Writing meaningless tests just to "hit lines"
- Testing trivial getters/setters

## 7️⃣ Coverage in the Titanic project

### What should be covered?

Focus on:
- feature engineering functions
- preprocessing logic
- pipeline construction
- error handling paths

Avoid forcing coverage on:
- visualization code
- logging
- CLI argument parsing (unless critical)

## 8️⃣ Increasing coverage: a meaningful example

Suppose coverage reports:

```
src/features.py 65% Missing: 70-85
```

These lines correspond to:
- rare branches,
- error handling,
- optional parameters.

Add a targeted test:

```python
def test_extract_title_missing_name():
    df = pd.DataFrame({"Name": [None]})
    out = extract_title(df)
    assert out["Title"].iloc[0] == "Unknown"

```

✔ Coverage increases
✔ Behavior is now explicitly defined

## 9️⃣ Skipping irrelevant files from coverage

You usually do not want coverage for:
- test files,
- configuration files,
- notebooks.

Create a `.coveragerc` file:

```ini
[run]
omit =
    tests/*
    */__init__.py
    *.ipynb
```

Now rerun:

```bash
coverage run -m pytest
coverage report -m
```

## 🔟 Coverage and Continuous Integration

In CI pipelines (e.g. GitHub Actions), coverage can be:
- reported automatically,
- enforced with thresholds,
- displayed with badges.

⚠️ In this course: Coverage is informative, not graded.

# Parametrization, Coverage, and CI Best Practices

## 1️⃣ Parametrization and test coverage

### How parametrization affects coverage

When you use `@pytest.mark.parametrize`, each parameter set executes the same code path with different inputs.

This has two important effects:
- ✔ increases behavioral confidence
- ✔ often increases coverage, because more branches are exercised

Example:

```python
@pytest.mark.parametrize("n_trees", [1, 5, 10, 50])
def test_pipeline_builds(n_trees):
    pipe = build_pipeline(n_trees)
    assert pipe is not None

```

This test:
- runs 4 times,
- executes the same lines,
- ensures the pipeline does not rely on a single hard-coded value.

⚠️ However: Running the same lines many times does not automatically increase coverage percentage. 

Coverage increases only if new lines or branches are executed.

### Example: parametrization that increases coverage

Suppose your code contains branching logic:

```python
def build_pipeline(n_trees):
    if n_trees <= 0:
        raise ValueError("n_trees must be positive")
    elif n_trees < 5:
        mode = "small"
    else:
        mode = "normal"
    return mode

```

A single test would not cover all branches.

Using parametrization:

```python
@pytest.mark.parametrize(
    "n_trees,expected",
    [
        (1, "small"),
        (5, "normal"),
        (20, "normal"),
    ],
)
def test_build_pipeline_modes(n_trees, expected):
    assert build_pipeline(n_trees) == expected

```

✔ All logical branches are now covered
✔ Coverage increases meaningfully

## 2️⃣ Good vs bad parametrization

Parametrization is powerful - but it can be misused.

### ❌ Bad parametrization (anti-pattern)

```python
@pytest.mark.parametrize("n_trees", range(1, 101))
def test_pipeline_many_values(n_trees):
    pipe = build_pipeline(n_trees)
    assert pipe is not None

```

Why this is bad:
- tests the same behavior 100 times
- slows down CI
- adds no new information
- inflates test runtime without increasing coverage

### ✅ Good parametrization (focused and meaningful)

```python
@pytest.mark.parametrize(
    "n_trees",
    [1, 5, 20],
)
def test_pipeline_supported_ranges(n_trees):
    pipe = build_pipeline(n_trees)
    assert pipe is not None

```

Why this is good:
- tests boundary cases
- tests representative values
- keeps CI fast
- communicates intent clearly

🧠 **Rule of thumb**: Parametrize behaviors, not numbers.

Use parametrization to:
- test categories of inputs,
- validate edge cases,
- exercise branches.

## 3️⃣ Parametrization in Continuous Integration (CI)

### Why CI makes parametrization even more important

In CI:
- tests run automatically,
- resources are limited,
- execution time matters.

Parametrization helps CI by:
- reducing duplicated test code,
- making failures easier to diagnose,
- ensuring consistency across configurations.

### Example: parametrized CI-safe test (Titanic)

```python
@pytest.mark.parametrize("n_trees", [5, 10])
def test_training_score_range(titanic_sample, n_trees):
    X, y = titanic_sample
    score = train_and_evaluate(X, y, n_trees)
    assert 0.0 <= score <= 1.0, "Score must be a valid accuracy"

```

Why this works well in CI:
- limited number of runs
- deterministic inputs
- no heavy computation
- clear failure messages

### What NOT to parametrize in CI

Avoid parametrizing:
- full model training loops,
- large datasets,
- random seeds without fixing them,
- external resources.

Instead:
- mock heavy operations,
- keep CI tests fast and deterministic.

## 4️⃣ Parametrization + coverage + CI: the right mindset

| Tool | Purpose |
|------|---------|
| Parametrization | Test multiple behaviors concisely |
| Coverage | Identify untested logic |
| CI | Enforce automatic verification |

They work together, but none replaces the others.